# Treinamento LoRA do Triunfo

Execute no Colab com T4, em uma sessão nova. As cinco etapas restauram o recorte preparado, configuram o modelo, mascaram o contexto e treinam com checkpoints no Drive. Não são necessários os pilotos históricos nem uma geração de baseline antes do treino.

Esta é uma consolidação das células executadas. Suas saídas estão em reports/; o notebook reorganizado não foi retreinado depois da documentação. Uma execução nova pode produzir pesos numericamente diferentes. Não sobrescreva um adaptador concluído.


## 1. Conectar o Google Drive


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

meu_drive = Path("/content/drive/MyDrive")
assert meu_drive.is_dir(), "O Google Drive não ficou acessível."

PASTA_PERSISTENTE = meu_drive / "Triunfo"
PASTA_PERSISTENTE.mkdir(parents=True, exist_ok=True)

print("Pasta persistente disponível:", PASTA_PERSISTENTE)

## 2. Restaurar dados e conferir o ambiente


In [ ]:
import hashlib
import io
import json
import platform
import tempfile
import zipfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import torch
from google.colab import files

PASTA_PERSISTENTE = Path("/content/drive/MyDrive/Triunfo")
assert PASTA_PERSISTENTE.is_dir(), "Conecte o Drive primeiro."

# Confirma escrita e leitura no Drive.
with tempfile.NamedTemporaryFile(
    dir=PASTA_PERSISTENTE, suffix=".txt", delete=False
) as arquivo:
    arquivo.write(b"triunfo: verificacao de escrita")
    verificacao = Path(arquivo.name)

assert verificacao.read_bytes() == b"triunfo: verificacao de escrita"
verificacao.unlink()
print("Escrita e leitura no Drive: conferidas")

nome_zip = "triunfo-dados-6bda777-seed42.zip"
zip_drive = PASTA_PERSISTENTE / nome_zip

if zip_drive.is_file():
    conteudo_zip = zip_drive.read_bytes()
else:
    print("\nSelecione o ZIP dos dados preparados:")
    enviados = files.upload()
    assert len(enviados) == 1, "Selecione somente o ZIP."
    conteudo_zip = next(iter(enviados.values()))

hashes_esperados = {
    "train.jsonl": "0a139d3a1b4eb5f0d2c7f854f6e5940570769e19e053f6950290e4ee2314c003",
    "validation.jsonl": "cef828a8d7ca10e0988cc16d056c571e10f13caf6c87458074bae10c2766b954",
    "test.jsonl": "bf4fb9eb1316dfc621eb1b1c1085fbb44d1750b017e5eb65104240bf420f356f",
}

with zipfile.ZipFile(io.BytesIO(conteudo_zip)) as backup:
    restaurados = {
        nome: backup.read(nome)
        for nome in [*hashes_esperados, "manifest.json"]
    }

for nome, esperado in hashes_esperados.items():
    assert hashlib.sha256(restaurados[nome]).hexdigest() == esperado, (
        f"Conteúdo diferente do original: {nome}"
    )
    print(f"{nome}: hash conferido")

manifesto_dados = json.loads(restaurados["manifest.json"])
assert manifesto_dados["dataset_revision"] == (
    "6bda777c88d21e5a204703c1ee45597a8fa4f734"
)

zip_drive.write_bytes(conteudo_zip)
assert hashlib.sha256(zip_drive.read_bytes()).digest() == (
    hashlib.sha256(conteudo_zip).digest()
)
print("\nZIP preservado no Drive:", zip_drive)

PASTA = Path("/content/triunfo_data")
PASTA.mkdir(parents=True, exist_ok=True)
for nome, conteudo in restaurados.items():
    (PASTA / nome).write_bytes(conteudo)

dados_treino = [
    json.loads(linha)
    for linha in restaurados["train.jsonl"].decode("utf-8").splitlines()
]
dados_validacao = [
    json.loads(linha)
    for linha in restaurados["validation.jsonl"].decode("utf-8").splitlines()
]
assert len(dados_treino) == 4469
assert len(dados_validacao) == 559

print("\nTreino:", len(dados_treino))
print("Validação:", len(dados_validacao))
print("Teste: preservado, sem carregar os exemplos")

print("\nAMBIENTE:")
print("Python:", platform.python_version())
for pacote in ["torch", "transformers", "peft", "accelerate", "datasets", "torchao"]:
    try:
        print(f"{pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"{pacote}: não instalado")

print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memória total:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GiB",
    )

## 3. Carregar o modelo e configurar LoRA


In [ ]:
import sys
import subprocess
import importlib
import json
from pathlib import Path

assert "modelo_lora" not in globals(), (
    "Já existe um modelo LoRA nesta sessão. Não carregue outro."
)

# Repete a correção já verificada na sessão anterior.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import (
    LoraConfig, TaskType, get_peft_model,
    get_peft_model_state_dict,
)

assert torch.cuda.is_available(), "A GPU precisa estar ativa."

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
    token=False,
)

print("Carregando o Qwen original na GPU do Colab...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
    use_safetensors=True,
    trust_remote_code=False,
    token=False,
)
model.requires_grad_(False)

set_seed(42)
config_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    init_lora_weights=True,
)

modelo_lora = get_peft_model(model, config_lora)
modelo_lora.eval()

treinaveis = {
    nome: parametro
    for nome, parametro in modelo_lora.named_parameters()
    if parametro.requires_grad
}
assert all("lora_" in nome for nome in treinaveis)
assert sum(p.numel() for p in treinaveis.values()) == 4358144

# Preserva o ponto inicial para conferir antes de iniciar o treino.
estado_inicial = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in get_peft_model_state_dict(modelo_lora).items()
}

resumo_recuperacao = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "rank": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "trainable_parameters": sum(p.numel() for p in treinaveis.values()),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_allocated_gib": round(
        torch.cuda.memory_allocated() / 1024**3, 2
    ),
    "optimizer_steps": 0,
}

destino = Path("/content/drive/MyDrive/Triunfo/recovery_setup.json")
destino.write_text(
    json.dumps(resumo_recuperacao, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(resumo_recuperacao, ensure_ascii=False, indent=2))
print("Configuração preservada no Drive:", destino)

## 4. Preparar os tokens supervisionados


In [ ]:
import json
from pathlib import Path

INSTRUCAO_BASE = (
    "You are a tool-use assistant. Use only the available tools. "
    "When calling a tool, return exactly one call inside square brackets: "
    "[tool_name(argument=value)]. Copy the tool name exactly, including "
    "spaces or dots. Use Python literal syntax for argument values. "
    "If no available tool can fulfill the request, explain why. "
    "If required information is missing, ask for it. "
    "Do not invent missing information or tool results."
)

def montar_mensagens(exemplo, incluir_resposta=False):
    ferramentas = json.dumps(
        exemplo["ferramentas"], ensure_ascii=False, sort_keys=True
    )
    mensagens = [
        {
            "role": "system",
            "content": INSTRUCAO_BASE + "\n\nAvailable tools:\n" + ferramentas,
        },
        {"role": "user", "content": exemplo["pedido"]},
    ]
    if incluir_resposta:
        mensagens.append({
            "role": "assistant",
            "content": exemplo["resposta"],
        })
    return mensagens

exemplos_treino_tokenizados = []

for indice, exemplo in enumerate(dados_treino, start=1):
    contexto = tokenizer.apply_chat_template(
        montar_mensagens(exemplo),
        tokenize=False,
        add_generation_prompt=True,
    )
    completo = tokenizer.apply_chat_template(
        montar_mensagens(exemplo, incluir_resposta=True),
        tokenize=False,
        add_generation_prompt=False,
    )

    contexto_ids = tokenizer(
        contexto, add_special_tokens=False, truncation=False
    )["input_ids"]
    completo_ids = tokenizer(
        completo, add_special_tokens=False, truncation=False
    )["input_ids"]

    n = len(contexto_ids)
    assert completo_ids[:n] == contexto_ids
    assert n < len(completo_ids) <= 2048

    exemplos_treino_tokenizados.append({
        "input_ids": completo_ids,
        "attention_mask": [1] * len(completo_ids),
        "labels": [-100] * n + completo_ids[n:],
    })

    if indice % 1000 == 0:
        print(f"Preparados {indice}/{len(dados_treino)}")

assert len(exemplos_treino_tokenizados) == 4469
assert max(
    len(e["input_ids"]) for e in exemplos_treino_tokenizados
) == 1711

print("Preparação restaurada: 4469 exemplos, sem cortes.")
print("Próxima etapa: treinamento com salvamento no Drive.")


## 5. Treinar e preservar o adaptador


In [ ]:
import gc
import json
import hashlib
import zipfile
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq, set_seed,
)
from peft import get_peft_model_state_dict
from google.colab import files

set_seed(42)

raiz_drive = Path("/content/drive/MyDrive/Triunfo")
assert raiz_drive.is_dir(), "Conecte o Google Drive primeiro."

pasta_treino = raiz_drive / "runs" / "lora_epoch1_recovery"
pasta_final = pasta_treino / "adapter_final"

assert not pasta_final.exists(), "Já existe um adaptador final nesta pasta."
assert not list(pasta_treino.glob("checkpoint-*")), (
    "Há checkpoints anteriores. Precisamos retomar o treino."
)
assert len(exemplos_treino_tokenizados) == 4469
assert all(
    torch.equal(t.detach().cpu(), estado_inicial[n])
    for n, t in get_peft_model_state_dict(modelo_lora).items()
), "Os adaptadores não estão no estado inicial."

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

colador = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# Confere que o preenchimento preserva a máscara da resposta.
amostra = colador(exemplos_treino_tokenizados[:2])
for i, original in enumerate(exemplos_treino_tokenizados[:2]):
    n = len(original["labels"])
    assert amostra["labels"][i, :n].tolist() == original["labels"]
    assert (amostra["labels"][i, n:] == -100).all().item()
del amostra

argumentos_treino = TrainingArguments(
    output_dir=str(pasta_treino),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=28,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=25,
    logging_nan_inf_filter=False,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none",
    seed=42,
    data_seed=42,
    dataloader_num_workers=0,
    push_to_hub=False,
)

pasta_treino.mkdir(parents=True, exist_ok=True)
(pasta_treino / "training_arguments.json").write_text(
    argumentos_treino.to_json_string(), encoding="utf-8"
)

modelo_lora.config.use_cache = False
trainer = Trainer(
    model=modelo_lora,
    args=argumentos_treino,
    train_dataset=Dataset.from_list(exemplos_treino_tokenizados),
    data_collator=colador,
    processing_class=tokenizer,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("Treino: 4469 exemplos | 1 época | 559 atualizações previstas")
print("Salvamentos no Drive:", pasta_treino)
resultado_treino = trainer.train()

# Salva o resultado final diretamente no Drive.
trainer.save_model(str(pasta_final))
tokenizer.save_pretrained(str(pasta_final))
trainer.save_state()
trainer.save_metrics("train", resultado_treino.metrics)

modelo_lora.gradient_checkpointing_disable()
modelo_lora.config.use_cache = True
modelo_lora.eval()

resumo_treino = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "training_examples": 4469,
    "global_step": trainer.state.global_step,
    "epoch": trainer.state.epoch,
    "metrics": resultado_treino.metrics,
    "gpu_peak_allocated_gib": round(
        torch.cuda.max_memory_allocated() / 1024**3, 2
    ),
    "adapter_path": str(pasta_final),
}
(pasta_treino / "training_epoch1.json").write_text(
    json.dumps(resumo_treino, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(resumo_treino, ensure_ascii=False, indent=2))

# Cria e confere o ZIP, também no Drive.
assert (pasta_final / "adapter_config.json").is_file()
assert (pasta_final / "adapter_model.safetensors").is_file()

arquivos = sorted(
    [p for p in pasta_final.rglob("*") if p.is_file()]
    + list(pasta_treino.glob("*.json"))
)
manifesto_backup = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "global_step": trainer.state.global_step,
    "files": {
        p.relative_to(pasta_treino).as_posix():
        hashlib.sha256(p.read_bytes()).hexdigest()
        for p in arquivos
    },
}

zip_final = pasta_treino / "triunfo-lora-epoch1-backup.zip"
with zipfile.ZipFile(zip_final, "w", zipfile.ZIP_DEFLATED) as z:
    for p in arquivos:
        z.write(p, p.relative_to(pasta_treino).as_posix())
    z.writestr(
        "backup_manifest.json",
        json.dumps(manifesto_backup, indent=2),
    )

with zipfile.ZipFile(zip_final) as z:
    for nome, esperado in manifesto_backup["files"].items():
        assert hashlib.sha256(z.read(nome)).hexdigest() == esperado

print("Backup conferido e salvo no Drive:", zip_final)
files.download(str(zip_final))